***

Preparing Workspace

***

In [ ]:
export=False



import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
pd.options.display.float_format = '{:.2f}'.format


path_git     = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_config0 = path_git / 'config'
path_config  = path_git / 'Data' / 'Census' / 'config'

path_plots = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('Export Location: ' + str(path_plots))




path_func = path_config0 / 'Functions.py'
with path_func.open("r") as f:
    exec(f.read())

path_func = path_config0 / 'plot.py'
with path_func.open("r") as f:
    exec(f.read())




***

Pop_3

***

In [ ]:

# Set Indicator
indicator = 'Pop_3'
plot_name = 'race_ethnicity'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)



## Organizing ---

df_plot = df_mpo.copy()


df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] != 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'American Indian or Alaska Native (NH)'
    , 'Native Hawaiian or other Pacific Islander (NH)'
    , 'Some other race (NH)'
    , 'Two or more races (NH)'
    , 'Black or African American (NH)'
    , 'Asian (NH)'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values('Sort', ascending=False)
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'American Indian or Alaska Native (NH)': '#A97142'
    , 'Native Hawaiian or other Pacific Islander (NH)': '#006A4E'
    , 'Some other race (NH)': '#7E587E'
    , 'Two or more races (NH)': '#1F45FC'
    , 'Asian (NH)': '#9DC209'
    , 'Black or African American (NH)': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.bar(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', color_discrete_map=color_map)


title = '<b>Race and Ethnicity</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Pop_4

***

In [ ]:


# Set Indicator
indicator = 'Pop_4'
plot_name = 'age_groups'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)



## Organizing ---

df_plot = df_mpo.copy()


df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Under 18'
    , '18 to 64'
    , '65+'
])
    
df_plot = df_plot.sort_values('Sort', ascending=False)
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---
color_map = {
    'Under 18': '#9DC209'
    , '18 to 64': '#1E90FF'
    , '65+': '#1F45FC'
}


fig = px.bar(df_plot, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map)


title = '<b>Population Age Distribution</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Commute_1

***

In [ ]:
# Set Indicator
indicator = 'Commute_1'
plot_name = 'nondrive_peers'
geography = 'MSA'


## Importing ---


file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_msa = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_msa.copy()


df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot = df_plot[df_plot['Variable'].isin([
    'Public transportation (excluding taxicab)'
    , 'Bicycle'
    , 'Walked'
    , 'Other means'
    , 'Worked from home'
])]
df_plot['MSA'] = df_plot['MSA'].map(peer_msa_labels)




df_plot['total_pop'] = df_plot.groupby(['Year', 'MSA_ID', 'Race_Ethnicity'])['Percentage'].transform('sum')


df_plot['Variable_sort'] = pd.Categorical(df_plot['Variable'], [
    'Worked from home'
    , 'Other means'
    , 'Walked'
    , 'Bicycle'
    , 'Public transportation (excluding taxicab)'
])

    
df_plot = df_plot.sort_values(['total_pop', 'Variable_sort'], ascending = [True, True])
df_plot = df_plot.drop(['total_pop', 'Variable_sort'], axis = 1)

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---
color_map = {
    'Public transportation (excluding taxicab)':'#7E587E'
       , 'Bicycle':"#FBB117"
       , 'Walked':"#9DC209"
       , 'Other means': '#1E90FF'
       , 'Worked from home':'#1F45FC'
}


fig = px.bar(df_plot, x='Percentage', y='MSA', color='Variable', color_discrete_map=color_map, orientation='h')


ticktext = []
for geography in df_plot['MSA'].unique():
    if geography in ['Sacramento, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)
        
fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['MSA'].unique(), ticktext=ticktext))


title = f'<b>Non-driving Modes of Commute: Peer Region Comparison, {df_plot.Year.max()}</b>'
fig.update_xaxes(tick0=0, dtick=5, ticksuffix='%', range = [0, 31])
fig.update_traces(hovertemplate='%{x}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.15,xanchor="right", x=0.825))


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Commute_1'
plot_name = 'commute_modes'


## Importing ---

geography = 'MSA'
file_msa = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_msa = pd.read_excel(file_msa, sheet_name=geography)

geography = 'MPO'
file_mpo = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_mpo, sheet_name=geography)



## Organizing ---
df_plot = df_mpo.copy()


df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']

display(df_plot.head())



## Plotting ---
color_map = {
    'Car, truck, or van Drove alone':"#1F45FC"
       , 'Car, truck, or van Carpooled':"#1E90FF"
       , 'Public transportation (excluding taxicab)':'#7E587E'
       , 'Bicycle':"#DC381F"
       , 'Walked':"#FBB117"
       , 'Other means': '#9DC209'
       , 'Worked from home':'#006A4E'
}


fig = px.bar(df_plot, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map)


title = '<b>Commute Mode by Category</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, ticksuffix='%', range = [0, 102])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Commute_1'
plot_name = 'nondrive_modes_years'


## Importing ---

geography = 'MSA'
file_msa = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_msa = pd.read_excel(file_msa, sheet_name=geography)

geography = 'MPO'
file_mpo = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_mpo, sheet_name=geography)



## Organizing ---

df_plot = df_mpo.copy()

df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Variable'].isin([
    'Public transportation (excluding taxicab)'
    , 'Bicycle'
    , 'Walked'
    , 'Other means'
    , 'Worked from home'
])]


display(df_plot.head())


## Plotting ---

color_map = {
    'Public transportation (excluding taxicab)':'#7E587E'
       , 'Bicycle':"#FBB117"
       , 'Walked':"#9DC209"
       , 'Other means': '#1E90FF'
       , 'Worked from home':'#1F45FC'
}

fig = px.line(df_plot, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map, markers=True)

title = '<b>Non-Drive Commute Modes</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=5, ticksuffix='%', range = [0, 21])
fig.update_xaxes(tick0=0, dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.175,xanchor="right", x=0.7))#0.75


plot_agol(export=export)


***

Edu_1

***

In [ ]:
# Set Indicator
indicator = 'Edu_1'
plot_name = '2022'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Total Less than high school diploma'
    , 'Total High school graduate or GED'
    , "Total Some college or associate's degree"
    , "Total Bachelor's degree or higher"
])
    
df_plot = df_plot.sort_values(['Sort'], ascending = [False])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---
color_map  = {
    'Total Less than high school diploma':'#FBB117'
    , 'Total High school graduate or GED':'#9DC209'
    , "Total Some college or associate's degree":'#1E90FF'
    , "Total Bachelor's degree or higher":'#1F45FC'
}


fig = px.bar(df_plot, y='Percentage', x='Race_Ethnicity'
             , color='Variable'
             , color_discrete_map=color_map)


title = f'<b>Educational Attainment by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Broadband_2

***

In [ ]:
# Set Indicator
indicator = 'Broadband_2'
plot_name = 'access'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organzing ---
df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Low Speed'
    , 'High Speed'
    , "No Internet or No Computer"
])
    
df_plot = df_plot.sort_values(['Sort'], ascending = [False])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---
color_map  = {
    'Low Speed':'#1F45FC'
    , 'High Speed':'#1E90FF'
    , "No Internet or No Computer":'#9DC209'
}


fig = px.bar(df_plot, y='Percentage', x='Race_Ethnicity'
             , color='Variable'
             , color_discrete_map=color_map
            , barmode='group'
            , text=df_plot['Percentage'].apply(lambda x: '{0:1.1f}%'.format(x))
            )


title = f'<b>Broadband Access by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0, 101])
fig.update_traces(hovertemplate='%{y}')
fig.update_traces(textfont_size=12, textposition="outside", cliponaxis=False)
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Health_1

***

In [ ]:
# Set Indicator
indicator = 'Health_1'
plot_name = 'grocery_access'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} CPS.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['PTDTRACE'].isin(['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)'])]

conditions = [
      (df_plot['HES1A'] == 'Yes'        ) & (df_plot['HES1B'] == 'Yes'        )
    , (df_plot['HES1A'] == 'Yes'        ) & (df_plot['HES1B'] == 'No'         )
    , (df_plot['HES1A'] == 'No'         ) & (df_plot['HES1B'] == 'Yes'        )
    , (df_plot['HES1A'] == 'No'         ) & (df_plot['HES1B'] == 'No'         )
    , (df_plot['HES1A'] == 'Yes'        ) & (df_plot['HES1B'] == 'No Response')
    , (df_plot['HES1A'] == 'No Response') & (df_plot['HES1B'] == 'Yes'        )
    , (df_plot['HES1A'] == 'No'         ) & (df_plot['HES1B'] == 'No Response')
    , (df_plot['HES1A'] == 'No Response') & (df_plot['HES1B'] == 'No'         )
    , (df_plot['HES1A'] == 'No Response') & (df_plot['HES1B'] == 'No Response')
]

choices = ['Yes for both Grocery and Convenience'
           , 'Yes for Grocery, No for Convenience'
           , 'No for Grocery, Yes for Convenience'
           , 'No for Grocery, No for Convenience'
           , 'Yes for Grocery, No response for Convenience'
           , 'No response for Grocery, Yes for Convenience'
           , 'No for Grocery, No response for Convenience'
           , 'No response for Grocery, No for Convenience'
           , 'No response for Grocery, No response for Convenience'
           ]

df_plot['Survey'] = np.select(conditions, choices, default = 'No')


df_plot['Sort'] = pd.Categorical(df_plot['Survey'], [
    'Yes for both Grocery and Convenience'
       , 'Yes for Grocery, No for Convenience'
       , 'No for Grocery, Yes for Convenience'
       , 'No for Grocery, No for Convenience'
       , 'Yes for Grocery, No response for Convenience'
       , 'No response for Grocery, Yes for Convenience'
       , 'No for Grocery, No response for Convenience'
       , 'No response for Grocery, No for Convenience'
       , 'No response for Grocery, No response for Convenience'
])
    
df_plot = df_plot.sort_values(['Sort'], ascending = [True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---
color_map  = {
    'Yes for both Grocery and Convenience': '#1F45FC'
       , 'Yes for Grocery, No for Convenience': '#1E90FF'
       , 'No for Grocery, Yes for Convenience': '#9DC209'
       , 'No for Grocery, No for Convenience': '#DC381F'
       , 'Yes for Grocery, No response for Convenience': '#7E587E'
       , 'No response for Grocery, Yes for Convenience': '#9E7BFF'
       , 'No for Grocery, No response for Convenience': '#006A4E'
       , 'No response for Grocery, No for Convenience': '#E56717'
       , 'No response for Grocery, No response for Convenience': '#FBB117'
}



fig = px.bar(df_plot, y='Percentage', x='PTDTRACE'
             , color='Survey'
             , color_discrete_map=color_map)


title = f'<b>Access to Grocery Stores by Race/Ethnicity, {2009}-{2023}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Health_2

***

In [ ]:
# Set Indicator
indicator = 'Health_2'
plot_name = 'uninsured'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Variable'] == 'No health insurance coverage']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
df_plot['Percentage'] = round(df_plot['Percentage'], 1)
display(df_plot.head())


## Plotting ---

color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}

fig = px.line(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', markers = True,
              color_discrete_map=color_map)


title = '<b>Rate of People Without Health Insurance by Race/Ethnicity</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=5, ticksuffix='%', range = [0, 26])
fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Cost_3

***

In [ ]:


# Set Indicator
indicator = 'Cost_3'
plot_name = 'vacancy_rate'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Variable'] == 'Vacant']
df_plot = df_plot.reset_index(drop = True)
df_plot['Percentage'] = round(df_plot['Percentage'], 1)
display(df_plot.head())


## Plotting ---

fig = px.line(df_plot, x='Year', y='Percentage', markers = True)
fig['data'][0]['line']['color']='#1E90FF'

title = '<b>Unoccupied Housing Rate</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=0.5, ticksuffix='%', range = [1.75,5.25])
fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Cost_5

***

In [ ]:
# Set Indicator
indicator = 'Cost_5'
plot_name = 'owners'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
df_plot = df_plot[df_plot['Variable'] == 'Owner occupied']
df_plot['Percentage'] = round(df_plot['Percentage'], 1)
display(df_plot.head())


## Plotting ---

color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}

fig = px.line(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', markers = True,
              color_discrete_map=color_map)

title = '<b>Owner Occupied Housing by Race/Ethnicity</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=10, ticksuffix='%', range = [0, 100])
fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Cost_5'
plot_name = 'renters'
geography='MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()

race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Variable'] == 'Renter occupied']
df_plot['Percentage'] = round(df_plot['Percentage'], 1)
display(df_plot.head())


## Plotting ---

color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}

fig = px.line(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', markers = True,
              color_discrete_map=color_map)

title = '<b>Renter Occupied Housing by Race/Ethnicity</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,100])
fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Cost_6

***

In [ ]:
# Set Indicator
indicator = 'Cost_6'
plot_name = 'burden'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Housing Type'].isin(['Owner', 'Renter'])]

race_ethnicity = ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['RAC1P'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot = df_plot.groupby(['MPO', 'Year', 'RAC1P', 'Housing Burden'], as_index = False).agg(Total = ('Households', 'sum'))
df_plot['Percentage'] = 100*df_plot['Total'] / df_plot.groupby(['MPO', 'Year', 'RAC1P'])['Total'].transform('sum')
df_plot = df_plot[df_plot['Housing Burden'].isin(['Cost burden >30% to <=50%', 'Cost burden >50%'])]

display(df_plot.head())


df_plot['Housing_sort'] = pd.Categorical(df_plot['Housing Burden'], [
    'Cost burden >50%'
    , 'Cost burden >30% to <=50%'
])
    
df_plot = df_plot.sort_values(['Housing_sort'], ascending = [False])
df_plot = df_plot.drop(['Housing_sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)


## Plotting ---

color_map_cost  = {
    'Cost burden >50%': '#9DC209'
    , 'Cost burden >30% to <=50%': '#1E90FF'
}


fig = px.bar(df_plot, y='Percentage', x='RAC1P'
             , color='Housing Burden'
             , color_discrete_map=color_map_cost)


title = f'<b>Housing Cost Burden by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Income_1

***

In [ ]:

export=True

# Set Indicator
indicator = 'Income_1'
plot_name = 'msa'


## Importing ---

geography = 'MSA'
file_msa = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_msa = pd.read_excel(file_msa, sheet_name=geography)

geography = 'MPO'
file_mpo = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_mpo, sheet_name=geography)


## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['MSA'] = df_plot['MSA'].map(peer_msa_labels)


# df_plot['MSA_sort'] = pd.Categorical(df_plot['MSA'], sort_peers)
# df_plot = df_plot.sort_values('MSA_sort')
# df_plot = df_plot.drop(['MSA_sort'], axis = 1)

df_plot = df_plot.sort_values(['Median Household Income'], ascending=[False])

df_plot = df_plot.reset_index(drop=True)


display(df_plot.head())


## Plotting ---


fig = px.bar(df_plot, y='MSA', x='Median Household Income'
             , color='MSA'
             , color_discrete_map=color_map_nat_peers
             , orientation='h')

# fig['data'][0]['marker']['line']['color'] = '#000000'
# fig['data'][1]['marker']['line']['color'] = '#000000'
# fig['data'][0]['marker']['line']['width'] = 1
# fig['data'][1]['marker']['line']['width'] = 1

ticktext = []
for geography in df_plot['MSA'].unique():
    if geography in ['Sacramento, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)
        
fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['MSA'].unique(), ticktext=ticktext))

title = f'<b>Median Household Income by MSA (Metropolitan Statistical Area), {df_plot.Year.max()}<b>'
fig.update_xaxes(tick0=0, dtick=20000, tickprefix='$', tickformat = ',.0f')
fig.update_traces(hovertemplate='%{x}')
fig.update_layout(showlegend = False)


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Income_1'
plot_name = 'sacog'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---
fig = px.bar(df_plot, x='Year', y='Median Household Income')
fig.update_traces(marker_color='#1E90FF')


title = '<b>Real Household Income in Sacramento Region Through Time<b>'
fig.update_yaxes(tick0=0, dtick=10000, range=[0, 101000], tickprefix='$', tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend = False)


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Income_1'
plot_name = 'race_ethnicity'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()

race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

fig = px.bar(df_plot, x='Race_Ethnicity', y='Median Household Income')
fig.update_traces(marker_color='#1E90FF')

title = f'<b>Household Income by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20000, tickprefix='$', tickformat = ',.0f', range = [0, 121000])
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend = False)


plot_agol(export=export)


***

Income_2

***

In [ ]:
# Set Indicator
indicator = 'Income_2'
plot_name = 'income_brackets'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year'].isin([2016, 2023])]
df_plot['Year'] = df_plot['Year'].astype('str')
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
df_plot = df_plot.reset_index(drop = True)

df_plot['Income_sort'] = pd.Categorical(df_plot['Income Bracket'], [
    'Low Income'
    , 'Moderate Income'
    , 'High Income'
])
    
df_plot = df_plot.sort_values(['Income_sort', 'Year'], ascending = [True, True])
df_plot = df_plot.drop(['Income_sort'], axis = 1)

display(df_plot.head())


## Plotting ---

color_map = {
    '2016':'#79BAEC'
    , '2023':'#006A4E'
}

fig = px.bar(df_plot, y='Percentage', x='Income Bracket'
             , color='Year'
             , barmode='group'
             , color_discrete_map=color_map)


year_max = df_plot.Year.max()
year_min = df_plot.Year.min()

title = f'<b>Share of Household Income:  {year_min} to {year_max}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%', range = [0, 52])
fig.update_traces(hovertemplate='%{y}')

export=True
plot_agol(export=export)


***

Income_3

***

In [ ]:

# Set Indicator
indicator = 'Income_3'
plot_name = 'time_series'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---
df_plot = df_mpo.copy()


race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot['Percent of Regional Median Household Income'] = round(df_plot['Percent of Regional Median Household Income'], 1)
df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}

fig = px.line(df_plot, x='Year', y='Percent of Regional Median Household Income'
              , color='Race_Ethnicity'
              , color_discrete_map=color_map
              , markers=True)


title = '<b>Median Income by Race/Ethnicity compared to Overall Median Income </b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=10, ticksuffix='%', range = [55,125])
fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Income_4

***

In [ ]:
# Set Indicator
indicator = 'Income_4'
plot_name = 'poverty_rate'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()

race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot = df_plot[df_plot['Variable'] == 'Total Income in the past 12 months below poverty level']
df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---
fig = px.bar(df_plot, x='Race_Ethnicity', y='Percentage')
fig.update_traces(marker_color='#1E90FF')


title = f'<b>Relative Poverty by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=5, ticksuffix='%', range = [0,27])
fig.update_xaxes(dtick=1)
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Labor_1

***

In [ ]:
# Set Indicator
indicator = 'Labor_1'
plot_name = 'participation'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} ACS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()

race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage'], 1)


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Employed'
    , 'Not in Labor Force'
    , 'Unemployed'
])
    
df_plot = df_plot.sort_values('Sort', ascending = True)
df_plot = df_plot.drop(['Sort'], axis = 1)

display(df_plot.head())


df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
         "Employed":"#1E90FF",
         "Unemployed": "#9DC209",
         "Not in Labor Force": "#1F45FC"
}

fig = px.bar(df_plot, x='Race_Ethnicity', y='Percentage'
             , color = 'Variable'
             , color_discrete_map=color_map)



title = f'<b>Working Age (16-64) Labor Force Participation by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,102])
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


***

Jobs_4

***

In [ ]:


# Set Indicator
indicator = 'Jobs_4'
plot_name = 'new_firms'
geography = 'MSA'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} LEHD.xlsx"
df_msa = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_msa.copy()

df_plot = df_plot[df_plot['Firm Age'] == 'Less than or equal to 5 years old']
df_plot = df_plot[df_msa['Quarter'] == '2023-Q3']
df_plot = df_plot.reset_index(drop=True)
df_plot['MSA'] = df_plot['MSA'].map(peer_msa_labels)


df_plot = df_plot.sort_values('Percentage', ascending=False)
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)

display(df_plot.head())



## Plotting ---


fig = px.bar(df_plot, y='MSA', x='Percentage'
             , color='MSA'
             , color_discrete_map=color_map_nat_peers
             , orientation='h')

ticktext = []
for geography in df_plot['MSA'].unique():
    if geography in ['Sacramento, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)
        
fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['MSA'].unique(), ticktext=ticktext))


title = 'Percent of Employment in Firms 5 Years or Younger: Q3 2023'
fig.update_xaxes(tick0=0, dtick=5, ticksuffix='%')
fig.update_traces(hovertemplate='Firm Age 5 Years or Less: %{x}')
fig.update_layout(showlegend = False)

export=False
plot_agol(export=export)



***

Accessibility_1

***

In [ ]:
# Set Indicator
indicator = 'Accessibility_1'
plot_name = 'race_ethnicity'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


race_ethnicity = ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['RAC1P'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)

df_plot = df_plot[df_plot['JWTRNS'].isin([
    'Public transportation (bus, subway, or rail)'
    , 'Bicycle'
    , 'Walked'
    , 'Other method'
    , 'Worked from home'
])]


df_plot['Sort'] = pd.Categorical(df_plot['JWTRNS'], [
    'Public transportation (bus, subway, or rail)'
    , 'Bicycle'
    , 'Walked'
    , 'Other method'
    , 'Worked from home'
])
    
df_plot = df_plot.sort_values(['Sort', 'RAC1P'], ascending = [False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)



df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
    'Public transportation (bus, subway, or rail)':'#7E587E'
       , 'Bicycle':"#FBB117"
       , 'Walked':"#9DC209"
       , 'Other method': '#1E90FF'
       , 'Worked from home':'#1F45FC'
}


fig = px.bar(df_plot, x='RAC1P', y='Percentage'
             , color = 'JWTRNS'
             , color_discrete_map=color_map)



title = f'<b>Alternative Modes of Commute by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=5, ticksuffix='%', range = [0,31])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Accessibility_2

***

In [ ]:


# Set Indicator
indicator = 'Accessibility_2'
plot_name = 'income'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)

df_plot = df_plot[df_plot['JWTRNS'].isin([
    'Car, truck, or van'
    , 'Public transportation (bus, subway, or rail)'
    , 'Bicycle'
    , 'Walked'
    , 'Other method'
    , 'Worked from home'
])]


df_plot['Sort1'] = pd.Categorical(df_plot['Income Bracket'], [
    'Low Income'
    , 'Moderate Income'
    , 'High Income'
])

df_plot['Sort2'] = pd.Categorical(df_plot['JWTRNS'], [
    'Car, truck, or van'
    , 'Public transportation (bus, subway, or rail)'
    , 'Bicycle'
    , 'Walked'
    , 'Other method'
    , 'Worked from home'
])
    
df_plot = df_plot.sort_values(['Sort1', 'Sort2'], ascending = [True, False])
df_plot = df_plot.drop(['Sort1', 'Sort2'], axis = 1)



df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
    'Car, truck, or van':'#DC381F'
       , 'Public transportation (bus, subway, or rail)':'#7E587E'
       , 'Bicycle':"#FBB117"
       , 'Walked':"#9DC209"
       , 'Other method': '#1E90FF'
       , 'Worked from home':'#1F45FC'
}


fig = px.bar(df_plot, x='Income Bracket', y='Percentage'
             , color = 'JWTRNS'
             , color_discrete_map=color_map)


title = f'<b>Commute Mode by Income, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,101])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(fig, export, title, indicator, plot_name, path_plots)


***

Accessibility_3

***

In [ ]:


# Set Indicator
indicator = 'Accessibility_3'
plot_name = 'auto_ownership'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


race_ethnicity = ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['RAC1P'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)


df_plot['Sort'] = pd.Categorical(df_plot['VEH'], [
    'No vehicles'
    , '3 or more vehicles'
    , '2 vehicles'
    , '1 vehicle'
])
    
df_plot = df_plot.sort_values(['Sort', 'RAC1P'], ascending = [False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
       'No vehicles':"#FBB117"
       , '3 or more vehicles':"#9DC209"
       , '2 vehicles': '#1E90FF'
       , '1 vehicle':'#1F45FC'
}


fig = px.bar(df_plot, x='RAC1P', y='Percentage'
             , color = 'VEH'
             , color_discrete_map=color_map)


title = f'<b>Auto Ownership by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,101])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Accessibility_4

***

In [ ]:
# Set Indicator
indicator = 'Accessibility_4'
plot_name = 'travel_time_race_ethnicity'
geography = 'MPO'

## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


race_ethnicity = ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['RAC1P'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)


df_plot['Sort'] = pd.Categorical(df_plot['Travel Time'], [
    'No commute (worked from home)'
    , '0 to 15 minutes'
    , '15 to 30 minutes'
    , 'More than 30 minutes'
])
    
df_plot = df_plot.sort_values(['Sort', 'RAC1P'], ascending = [False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
    'No commute (worked from home)':"#FBB117"
       , '0 to 15 minutes':"#9DC209"
       , '15 to 30 minutes': '#C11B17'
       , 'More than 30 minutes':'#1F45FC'
}


fig = px.bar(df_plot, x='RAC1P', y='Percentage'
             , color = 'Travel Time'
             , color_discrete_map=color_map)



title = f'<b>Alternative Modes of Commute by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,100])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Accessibility_4'
plot_name = 'travel_time_income'
geography = 'MPO'


## Importing ---

file_name = path_plots / 'Data' / f"{indicator} {geography} PUMS5.xlsx"
df_mpo = pd.read_excel(file_name, sheet_name=geography)


## Organizing ---

df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['RAC1P'] == 'All']
df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)


df_plot['Sort1'] = pd.Categorical(df_plot['Income Bracket'], [
    'Low Income'
    , 'Moderate Income'
    , 'High Income'
])

df_plot['Sort2'] = pd.Categorical(df_plot['Travel Time'], [
    'No commute (worked from home)'
    , '0 to 15 minutes'
    , '15 to 30 minutes'
    , 'More than 30 minutes'
])
    
df_plot = df_plot.sort_values(['Sort1', 'Sort2'], ascending = [True, False])
df_plot = df_plot.drop(['Sort1', 'Sort2'], axis = 1)

df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
    'No commute (worked from home)':"#FBB117"
       , '0 to 15 minutes':"#9DC209"
       , '15 to 30 minutes': '#C11B17'
       , 'More than 30 minutes':'#1F45FC'
}


fig = px.bar(df_plot, x='Income Bracket', y='Percentage'
             , color = 'Travel Time'
             , color_discrete_map=color_map)


title = f'<b>Alternative Modes of Commute by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,100])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)
